# Language Translation Tool
#### LinguaAI Translator is an AI-powered multilingual translation system developed as part of the CodeAlpha Internship Program, designed to break language barriers through fast and intelligent real-time translation.

#### By combining advanced language processing, auto-detection, text-to-speech, and an interactive user interface, this project delivers a seamless communication experience across multiple global languages.


### Install Required libraries and Dependencies


In [1]:
!pip install deep-translator langdetect gtts -q
print(" All libraries installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
 All libraries installed successfully!


### Import Essential Libraries & Setup

In [2]:
import warnings
warnings.filterwarnings('ignore')

from deep_translator import GoogleTranslator
from langdetect import detect, DetectorFactory
from gtts import gTTS
from IPython.display import display, HTML, Audio, Javascript
from IPython.display import Audio, display
import ipywidgets as widgets
import base64
import io
import datetime

# Make language detection consistent
DetectorFactory.seed = 0

print(" All imports successful!")

 All imports successful!


### Language Configuration

In [3]:
# Comprehensive language list with codes
LANGUAGES = {
    'Auto Detect': 'auto',
    'Afrikaans': 'af',
    'Albanian': 'sq',
    'Arabic': 'ar',
    'Azerbaijani': 'az',
    'Bengali': 'bn',
    'Bosnian': 'bs',
    'Bulgarian': 'bg',
    'Catalan': 'ca',
    'Chinese (Simplified)': 'zh-CN',
    'Chinese (Traditional)': 'zh-TW',
    'Croatian': 'hr',
    'Czech': 'cs',
    'Danish': 'da',
    'Dutch': 'nl',
    'English': 'en',
    'Estonian': 'et',
    'Finnish': 'fi',
    'French': 'fr',
    'Galician': 'gl',
    'Georgian': 'ka',
    'German': 'de',
    'Greek': 'el',
    'Gujarati': 'gu',
    'Haitian Creole': 'ht',
    'Hebrew': 'iw',
    'Hindi': 'hi',
    'Hungarian': 'hu',
    'Icelandic': 'is',
    'Indonesian': 'id',
    'Irish': 'ga',
    'Italian': 'it',
    'Japanese': 'ja',
    'Kannada': 'kn',
    'Korean': 'ko',
    'Latin': 'la',
    'Latvian': 'lv',
    'Lithuanian': 'lt',
    'Macedonian': 'mk',
    'Malay': 'ms',
    'Maltese': 'mt',
    'Marathi': 'mr',
    'Norwegian': 'no',
    'Persian': 'fa',
    'Polish': 'pl',
    'Portuguese': 'pt',
    'Punjabi': 'pa',
    'Romanian': 'ro',
    'Russian': 'ru',
    'Serbian': 'sr',
    'Slovak': 'sk',
    'Slovenian': 'sl',
    'Spanish': 'es',
    'Swahili': 'sw',
    'Swedish': 'sv',
    'Tamil': 'ta',
    'Telugu': 'te',
    'Thai': 'th',
    'Turkish': 'tr',
    'Ukrainian': 'uk',
    'Urdu': 'ur',
    'Vietnamese': 'vi',
    'Welsh': 'cy',
}

# Map language detect codes to language names
CODE_TO_NAME = {v: k for k, v in LANGUAGES.items() if k != 'Auto Detect'}

print(f" {len(LANGUAGES)-1} languages loaded!")

 62 languages loaded!


### Translation Engine


In [ ]:
# Translation history storage
translation_history = []

def detect_language(text):
    """Detect the language of input text."""
    try:
        code = detect(text)
        name = CODE_TO_NAME.get(code, f'Unknown ({code})')
        return code, name
    except:
        return 'en', 'English'

def translate_text(text, source_lang, target_lang):
    """
    Translate text using Google Translate via deep-translator.
    Returns: (translated_text, detected_lang_name, status_message)
    """
    if not text.strip():
        return '', '', ' Please enter some text to translate.'

    try:
        detected_code = None
        detected_name = None

        if source_lang == 'auto':
            detected_code, detected_name = detect_language(text)
            src = detected_code
        else:
            src = source_lang

        if src == target_lang:
            return text, detected_name, ' Source and target languages are the same.'

        translator = GoogleTranslator(source=src, target=target_lang)
        translated = translator.translate(text)

        # Save to history
        history_entry = {
            'time': datetime.datetime.now().strftime('%H:%M:%S'),
            'original': text[:60] + '...' if len(text) > 60 else text,
            'translated': translated[:60] + '...' if len(translated) > 60 else translated,
            'from': detected_name or CODE_TO_NAME.get(src, src),
            'to': CODE_TO_NAME.get(target_lang, target_lang)
        }
        translation_history.insert(0, history_entry)
        if len(translation_history) > 10:
            translation_history.pop()

        status = f' Translated successfully! | {len(text)} chars input → {len(translated)} chars output'
        return translated, detected_name, status

    except Exception as e:
        return '', '', f' Translation failed: {str(e)}'


def text_to_speech(text, lang_code):
    """
    Convert text to speech and return base64 audio.
    """
    try:
        if not text.strip():
            return None
        # Use simple lang code (strip region suffix)
        tts_lang = lang_code.split('-')[0]
        tts = gTTS(text=text, lang=tts_lang, slow=False)
        audio_buffer = io.BytesIO()
        tts.write_to_fp(audio_buffer)
        audio_buffer.seek(0)
        audio_data = audio_buffer.read()
        b64 = base64.b64encode(audio_data).decode('utf-8')
        return b64
    except Exception as e:
        print(f"TTS Error: {e}")
        return None

print(" Translation engine ready!")

 Translation engine ready!


### Launch the Translation App


In [5]:
lang_names = list(LANGUAGES.keys())
lang_codes = list(LANGUAGES.values())

# ── Widgets ─────────────────────────────────────────────────

title_html = widgets.HTML(value="""
<div style="
  background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
  border-radius: 16px;
  padding: 28px 32px;
  margin-bottom: 4px;
  font-family: 'Segoe UI', Arial, sans-serif;
  text-align: center;
">
  <div style="font-size: 48px; margin-bottom: 8px;">🤖</div>
  <h1 style="color: #e0e0e0; font-size: 28px; margin: 0 0 6px 0; font-weight: 700; letter-spacing: 1px;">
    LinguaAI Translator
  </h1>
  <p style="color: #a0b8d8; font-size: 14px; margin: 0; letter-spacing: 0.5px;">
    Powered by Google Translate &nbsp;|&nbsp; 60+ Languages &nbsp;|&nbsp; Text-to-Speech &nbsp;|&nbsp; Auto Detect
  </p>
</div>
""")

# Source language dropdown
src_label = widgets.HTML('<div style="font-family:Segoe UI,Arial,sans-serif; font-size:13px; font-weight:600; color:#555; margin-bottom:4px;">🔤 SOURCE LANGUAGE</div>')
src_dropdown = widgets.Dropdown(
    options=lang_names,
    value='Auto Detect',
    layout=widgets.Layout(width='100%', height='36px')
)

# Swap button
swap_btn = widgets.Button(
    description=' ⇄ Swap',
    layout=widgets.Layout(width='100px', height='36px', margin='22px 8px 0 8px'),
    style=widgets.ButtonStyle(button_color='#0f3460', font_color='white')
)

# Target language dropdown
tgt_label = widgets.HTML('<div style="font-family:Segoe UI,Arial,sans-serif; font-size:13px; font-weight:600; color:#555; margin-bottom:4px;">🎯 TARGET LANGUAGE</div>')
tgt_dropdown = widgets.Dropdown(
    options=[l for l in lang_names if l != 'Auto Detect'],
    value='French',
    layout=widgets.Layout(width='100%', height='36px')
)

# Input text area
input_label = widgets.HTML('<div style="font-family:Segoe UI,Arial,sans-serif; font-size:13px; font-weight:600; color:#555; margin-bottom:4px;">✏️ ENTER TEXT TO TRANSLATE</div>')
input_text = widgets.Textarea(
    placeholder='Type or paste your text here... (supports up to 5000 characters)',
    layout=widgets.Layout(width='100%', height='130px')
)

# Character counter
char_count = widgets.HTML('<div style="font-family:monospace; font-size:12px; color:#888; text-align:right;">0 / 5000 characters</div>')

# Translate button
translate_btn = widgets.Button(
    description='🌐  Translate Now',
    layout=widgets.Layout(width='200px', height='44px'),
    style=widgets.ButtonStyle(button_color='#0f3460', font_color='white', font_weight='bold')
)

# Clear button
clear_btn = widgets.Button(
    description='🗑️ Clear',
    layout=widgets.Layout(width='100px', height='44px'),
    style=widgets.ButtonStyle(button_color='#e74c3c', font_color='white')
)

# Status bar
status_bar = widgets.HTML('<div style="font-size:13px; color:#888; font-family:Segoe UI,Arial,sans-serif; padding: 4px 0;"> </div>')

# Output text area
output_label = widgets.HTML('<div style="font-family:Segoe UI,Arial,sans-serif; font-size:13px; font-weight:600; color:#555; margin-bottom:4px;">📄 TRANSLATION RESULT</div>')
output_text = widgets.Textarea(
    placeholder='Translation will appear here...',
    layout=widgets.Layout(width='100%', height='130px')
)
output_text.disabled = True

# Detected language display
detected_lang_html = widgets.HTML('')

# TTS button
tts_btn = widgets.Button(
    description='🔊 Listen',
    layout=widgets.Layout(width='120px', height='40px'),
    style=widgets.ButtonStyle(button_color='#27ae60', font_color='white')
)

# Copy note
copy_note = widgets.HTML('<div style="font-family:Segoe UI,Arial,sans-serif; font-size:12px; color:#888; margin-left:8px; margin-top:10px;">💡 Tip: Click in the result box and press Ctrl+A then Ctrl+C to copy the translation.</div>')

# Audio output
audio_out = widgets.Output()

# History section
history_html = widgets.HTML('')

# Divider
divider = widgets.HTML('<hr style="border:none; border-top:1px solid #e0e0e0; margin:16px 0;">')
divider2 = widgets.HTML('<hr style="border:none; border-top:1px solid #e0e0e0; margin:16px 0;">')

# ── Helper: render history ───────────────────────────────────

def render_history():
    if not translation_history:
        history_html.value = ''
        return
    rows = ''.join([
        f"""
        <tr style="border-bottom:1px solid #f0f0f0;">
          <td style="padding:8px 6px; color:#888; font-size:12px; white-space:nowrap;">{h['time']}</td>
          <td style="padding:8px 6px; color:#333; font-size:13px;">{h['original']}</td>
          <td style="padding:8px 6px; color:#555; font-size:11px; white-space:nowrap;">→ {h['to']}</td>
          <td style="padding:8px 6px; color:#0f3460; font-size:13px;">{h['translated']}</td>
        </tr>
        """ for h in translation_history
    ])
    history_html.value = f"""
    <div style="font-family:Segoe UI,Arial,sans-serif;">
      <div style="font-size:13px; font-weight:600; color:#555; margin-bottom:8px;">📜 RECENT TRANSLATIONS</div>
      <div style="border:1px solid #e8e8e8; border-radius:10px; overflow:hidden;">
        <table style="width:100%; border-collapse:collapse; font-family:Segoe UI,Arial,sans-serif;">
          <thead>
            <tr style="background:#f5f7fa;">
              <th style="padding:8px 6px; text-align:left; font-size:11px; color:#888;">TIME</th>
              <th style="padding:8px 6px; text-align:left; font-size:11px; color:#888;">ORIGINAL</th>
              <th style="padding:8px 6px; text-align:left; font-size:11px; color:#888;">TO</th>
              <th style="padding:8px 6px; text-align:left; font-size:11px; color:#888;">TRANSLATED</th>
            </tr>
          </thead>
          <tbody>{rows}</tbody>
        </table>
      </div>
    </div>
    """

# ── Event Handlers ───────────────────────────────────────────

def on_input_change(change):
    n = len(change['new'])
    color = '#e74c3c' if n > 4800 else '#888'
    char_count.value = f'<div style="font-family:monospace; font-size:12px; color:{color}; text-align:right;">{n} / 5000 characters</div>'

input_text.observe(on_input_change, names='value')


def on_translate(b):
    text = input_text.value.strip()
    if not text:
        status_bar.value = '<div style="font-size:13px; color:#e74c3c; font-family:Segoe UI,Arial,sans-serif;"> Please enter some text first!</div>'
        return

    status_bar.value = '<div style="font-size:13px; color:#f39c12; font-family:Segoe UI,Arial,sans-serif;">⏳ Translating...</div>'

    src_name = src_dropdown.value
    tgt_name = tgt_dropdown.value
    src_code = LANGUAGES[src_name]
    tgt_code = LANGUAGES[tgt_name]

    result, detected_name, status = translate_text(text, src_code, tgt_code)

    output_text.disabled = False
    output_text.value = result
    output_text.disabled = True

    status_color = '#27ae60' if result else '#e74c3c'
    status_bar.value = f'<div style="font-size:13px; color:{status_color}; font-family:Segoe UI,Arial,sans-serif;">{status}</div>'

    if detected_name and src_code == 'auto':
        detected_lang_html.value = f'''
        <div style="
          display:inline-block;
          background:#e8f4fd;
          border:1px solid #b3d9f5;
          border-radius:20px;
          padding:4px 14px;
          font-size:12px;
          color:#1a6fa8;
          font-family:Segoe UI,Arial,sans-serif;
          font-weight:600;
          margin-bottom:6px;
        "> Detected: {detected_name}</div>'''
    else:
        detected_lang_html.value = ''

    render_history()


def on_clear(b):
    input_text.value = ''
    output_text.disabled = False
    output_text.value = ''
    output_text.disabled = True
    status_bar.value = ''
    detected_lang_html.value = ''
    char_count.value = '<div style="font-family:monospace; font-size:12px; color:#888; text-align:right;">0 / 5000 characters</div>'
    with audio_out:
        audio_out.clear_output()


def on_swap(b):
    current_src = src_dropdown.value
    current_tgt = tgt_dropdown.value
    if current_src != 'Auto Detect':
        src_dropdown.value = current_tgt
        tgt_dropdown.value = current_src


def on_tts(b):

    text = output_text.value.strip()

    if not text:
        status_bar.value = '''
        <div style="font-size:13px; color:#e74c3c;">
         No translated text to speak!
        </div>
        '''
        return

    try:

        tgt_name = tgt_dropdown.value
        lang_code = LANGUAGES[tgt_name]

        # gtts supports base code
        lang_code = lang_code.split("-")[0].lower()

        status_bar.value = '''
        <div style="font-size:13px; color:#f39c12;">
        🔊 Generating speech...
        </div>
        '''

        # Generate speech
        fp = io.BytesIO()

        tts = gTTS(
            text=text,
            lang=lang_code,
            slow=False
        )

        tts.write_to_fp(fp)

        fp.seek(0)

        # Play audio
        with audio_out:

            audio_out.clear_output(wait=True)

            display(
                Audio(
                    data=fp.read(),
                    autoplay=True
                )
            )

        status_bar.value = '''
        <div style="font-size:13px; color:#27ae60;">
         Audio playing!
        </div>
        '''

    except Exception as e:

        with audio_out:
            audio_out.clear_output()

        status_bar.value = f'''
        <div style="font-size:13px; color:#e74c3c;">
         Audio Error: {str(e)}
        </div>
        '''

# Bind events
translate_btn.on_click(on_translate)
clear_btn.on_click(on_clear)
swap_btn.on_click(on_swap)
tts_btn.on_click(on_tts)

# ── Layout Assembly ───────────────────────────────────────────

lang_row = widgets.HBox([
    widgets.VBox([src_label, src_dropdown], layout=widgets.Layout(flex='1')),
    swap_btn,
    widgets.VBox([tgt_label, tgt_dropdown], layout=widgets.Layout(flex='1'))
])

btn_row = widgets.HBox([translate_btn, clear_btn], layout=widgets.Layout(gap='12px'))
tts_row = widgets.HBox([tts_btn, copy_note])

main_ui = widgets.VBox([
    title_html,
    widgets.HTML('<div style="height:12px"></div>'),
    lang_row,
    widgets.HTML('<div style="height:12px"></div>'),
    input_label,
    input_text,
    char_count,
    widgets.HTML('<div style="height:8px"></div>'),
    btn_row,
    status_bar,
    divider,
    detected_lang_html,
    output_label,
    output_text,
    widgets.HTML('<div style="height:8px"></div>'),
    tts_row,
    audio_out,
    divider2,
    history_html,
], layout=widgets.Layout(padding='12px 20px', max_width='820px'))

display(main_ui)
print(" Translation Tool is ready! Enter text above and click Translate Now.")



 Translation Tool is ready! Enter text above and click Translate Now.


### Batch Translation
#### (Bonus Feature)


In [6]:
# Batch translation — translate one text to multiple languages at once
def batch_translate(text, target_languages):
    """
    Translate text into multiple target languages.
    target_languages: list of language names from LANGUAGES dict
    """
    print(f"🌐 Translating: '{text}'")
    print("="*55)
    for lang_name in target_languages:
        lang_code = LANGUAGES.get(lang_name)
        if not lang_code or lang_code == 'auto':
            continue
        result, _, _ = translate_text(text, 'auto', lang_code)
        print(f"  {lang_name:<22}: {result}")
    print("="*55)

# Example: translate "Hello World" into 10 languages
batch_translate(
    "Hello World! Welcome to my translation app.",
    ['French', 'Spanish', 'German', 'Arabic', 'Urdu',
     'Hindi', 'Chinese (Simplified)', 'Japanese', 'Russian', 'Turkish']
)

🌐 Translating: 'Hello World! Welcome to my translation app.'
  French                : Bonjour le monde! Bienvenue sur mon application de traduction.
  Spanish               : ¡Hola Mundo! Bienvenido a mi aplicación de traducción.
  German                : Hallo Welt! Willkommen in meiner Übersetzungs-App.
  Arabic                : مرحبا بالعالم! مرحبًا بك في تطبيق الترجمة الخاص بي.
  Urdu                  : ہیلو ورلڈ! میری ترجمہ ایپ میں خوش آمدید۔
  Hindi                 : हैलो वर्ल्ड! मेरे अनुवाद ऐप में आपका स्वागत है।
  Chinese (Simplified)  : 
  Japanese              : 「こんにちは世界」私の翻訳アプリへようこそ。
  Russian               : Привет, мир! Добро пожаловать в мое приложение для перевода.
  Turkish               : Selam Dünya! Çeviri uygulamama hoş geldiniz.
